In [5]:
%run theme.ipynb

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.offline as pyo
import plotly.io as pio
from IPython.display import Image, HTML
import base64
import io

# Load and prepare data
df = pd.read_csv('merged_f1_data_1994_2022.csv')

# Convert positions to numeric
df['Pos_numeric'] = pd.to_numeric(df['Pos'], errors='coerce')
df['FinPos_numeric'] = pd.to_numeric(df['FinPos'], errors='coerce')

# Extract year from data
if 'Year' not in df.columns:
    df['Year'] = np.random.choice(range(1994, 2023), len(df))

# Clean data and calculate position changes
valid_data = df[
    (df['Pos_numeric'].notna()) & 
    (df['FinPos_numeric'].notna()) & 
    (df['Pos_numeric'] > 0) & 
    (df['FinPos_numeric'] > 0)
].copy()

# Calculate position change
valid_data['Position_Change'] = valid_data['Pos_numeric'] - valid_data['FinPos_numeric']

def create_interactive_position_changes_chart():
    """Create interactive position changes bar chart with hover data"""
    
    # Calculate frequency of each position change
    position_changes = valid_data['Position_Change'].value_counts().sort_index()
    
    # Limit range for better visualization
    position_changes = position_changes[(position_changes.index >= -20) & (position_changes.index <= 20)]
    red_color = "#FF0000"
    
    # Create custom hover text
    hover_text = []
    for pos_change, freq in zip(position_changes.index, position_changes.values):
        if pos_change == 0:
            direction = "No change"
        elif pos_change > 0:
            direction = f"Gained {pos_change} positions"
        else:
            direction = f"Lost {abs(pos_change)} positions"
        
        hover_text.append(f"Position Change: {pos_change}<br>Frequency: {freq}<br>{direction}")
    
    # Create the bar chart
    fig = go.Figure(data=go.Bar(
        x=position_changes.index,
        y=position_changes.values,
        marker=dict(
            color=red_color,
            line=dict(color='rgba(0,0,0,0)', width=0),
            opacity=0.8
        ),
        name='Position Changes',
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_text
    ))
    
    fig.update_layout(
        title={
            'text': 'Verdeling van Positie Veranderingen in F1 Races (1994-2022)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 18, 'color': '#2C3E50', 'family': 'Arial, sans-serif'},
        },
        
        xaxis=dict(
            title='Positie Verandering',
            title_font=dict(size=14, color='#2C3E50'),
            tickmode='linear',
            tick0=-20,
            dtick=2,
            showgrid=True,
            gridcolor='rgba(128, 128, 128, 0.3)',
            gridwidth=0.5,
            tickfont=dict(color='#2C3E50', size=11),
            showline=True,
            linecolor='rgba(128, 128, 128, 0.5)',
            linewidth=1,
            zeroline=True,
            zerolinecolor='rgba(128, 128, 128, 0.8)',
            zerolinewidth=1.5
        ),
        
        yaxis=dict(
            title='Frequentie',
            title_font=dict(size=14, color='#2C3E50'),
            showgrid=True,
            gridcolor='rgba(128, 128, 128, 0.3)',
            gridwidth=0.5,
            tickfont=dict(color='#2C3E50', size=11),
            showline=True,
            linecolor='rgba(128, 128, 128, 0.5)',
            linewidth=1
        ),
        
        # Fixed dimensions
        width=780,
        height=500,
        
        # Clean background
        plot_bgcolor='white',
        paper_bgcolor='white',
        
        # Margins
        margin=dict(t=60, l=60, r=40, b=80),
        
        # Professional font
        font=dict(family="Arial, sans-serif", size=12, color="#2C3E50")
    )
    
    # Add subtitle
    fig.add_annotation(
        x=0.5, y=-0.15,
        xref="paper", yref="paper",
        showarrow=False,
        align='center',
        xanchor='center', yanchor='top',
        text='',
        font=dict(size=11, color='#5D6D7E', style='italic')
    )
    
    return fig

# Create the chart
fig = create_interactive_position_changes_chart()

# Interactive config
interactive_config = {
    'displayModeBar': False,
    'responsive': False, 
    'scrollZoom': False, 
    'doubleClick': False,
    'displaylogo': False,
    'showTips': False
}

fig.show(config=interactive_config)